## Dask-on-ray

https://docs.ray.io/en/latest/ray-more-libs/dask-on-ray.html

Ray maintains a Dask compatibility layer for using Ray’s scheduler with Dask-produced Task Graphs.

### Goals

1. Verify whether dask-on-ray works out of the box on “common lsdb workflows”.
2. Compare qualitative experience of using Dask’s dashboard against Ray’s.
3. Compare quantitative performance of Dask client performance against dask-on-ray, both timing and peak memory ideally.
4. If seeing instances where Ray seems valuable, capture in some kind of notebook for the docs.

In [1]:
import lsdb

### Version support

Version support is the following according to the documentation:

<img src="./assets/version-support.png" width=600/>

It looks outdated, so we will use the latest version of Dask unless we find problems.

In [2]:
lsdb.show_versions()


--------      SYSTEM INFO      --------
python        : 3.12.3
python-bits   : 64
OS            : Linux
OS-release    : 6.8.0-44-generic
Version       : #44-Ubuntu SMP PREEMPT_DYNAMIC Tue Aug 13 13:35:26 UTC 2024
machine       : x86_64
processor     : x86_64
byteorder     : little
LC_ALL        : 
LANG          : en_US.UTF-8
--------   INSTALLED VERSIONS   --------
lsdb          : 0.9.1.dev13+gbc10a6f4e
hats          : 0.9.1.dev5+ge1c6d8afe
nested-pandas : 0.6.9.dev35+ge699e7b9c
pandas        : 2.3.3
numpy         : 2.4.4
dask          : 2026.3.0
pyarrow       : 23.0.1
fsspec        : 2026.3.0


### How to install

Include the default extras to install the dependencies required for the Ray dashboard:

In [3]:
#%pip install -U ray[default] ipywidgets

### Basic usage

We can plan our Dask operations as usual:

In [4]:
# Select the first 10 partitions for testing
gaia = lsdb.open_catalog('/mnt/data/hats/catalogs/gaia_dr3').partitions[:10]

Then we just wrap the compute call on a dask-on-ray context:

In [5]:
import ray
from ray.util.dask import enable_dask_on_ray, disable_dask_on_ray

# Start the cluster.
# Details on how to configure the cluster in notebook 3.
ray.init(num_cpus=2)

# Wrap compute in a ray context:
enable_dask_on_ray()
try:
    gaia.compute()
finally:
    disable_dask_on_ray()
    # Close the cluster.
    ray.shutdown()

2026-04-10 13:00:47,910	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/dask/config.py:835: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(


Computing Catalog:   0%|          | 0/31 [00:00<?, ?it/s]

/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/dask/config.py:835: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(


We can also use a context manager...

```python
with enable_dask_on_ray():
    gaia.compute()
```

### Notes:

- Ray does not execute Dask tasks, so our Dask tasks need to be converted into Ray tasks.
- The tqdm progress bar displays 100% when the Ray tasks are submitted, not when the computation is finished.

Ray seems to report 31 tasks (~3 tasks per partition):

- It seems to be a non-optimized version of the task graph. 
- Or maybe related to the HTTP request issue that other folks reported was happening?

In [15]:
len(gaia._ddf.dask)

40

In [16]:
len(gaia._ddf.optimize().dask)

10

### Checking on progress

With the help of the dashboard we can see how many Ray tasks were submitted and their progress:

<img src="assets/dashboard-jobs.png" width=800>